# FlipAware Colab test

Runs the full pipeline on the `small` profile, one stage at a time, with a check after every stage.

Before you start:
1. Runtime > Change runtime type > **T4 GPU**
2. Add a Colab secret named `HF_TOKEN` (key icon on the left) with a HuggingFace **write** token, and turn on notebook access
3. If the GitHub repo is private, also add a secret `GH_TOKEN` with a GitHub token that can read it

Everything is saved to Google Drive, so if Colab disconnects, run cells 1 to 3 again and continue from the stage you were on. Finished work is skipped.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Drive, tokens, settings

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
try:
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
except Exception:
    os.environ["GH_TOKEN"] = ""
os.environ["FLIP_PROFILE"] = "small"
os.environ["FLIP_TRACKERS"] = "tensorboard,mlflow"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
print("ready")

## 3. Code and packages

In [ ]:
import os, subprocess
repo = "/content/flipaware"
if not os.path.isdir(repo):
    url = "https://github.com/amareshhebbar/flipaware.git"
    if os.environ.get("GH_TOKEN"):
        url = f"https://{os.environ['GH_TOKEN']}@github.com/amareshhebbar/flipaware.git"
    subprocess.run(["git", "clone", url, repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "pull"], check=True)
%cd /content/flipaware
!bash setup.sh
WORK = subprocess.run(["python", "-m", "flipaware.common", "workdir"], capture_output=True, text=True).stdout.strip()
os.environ["WORK"] = WORK
os.makedirs(f"{WORK}/logs", exist_ok=True)
print("work folder:", WORK)

## 4. Stage helper

Runs one stage, shows its output live, and saves it to the logs folder so stage 12 can publish it.

In [ ]:
import subprocess, sys

def stage(n, mod, *args):
    tag = f"{n:02d}_{mod}" + ("_" + "".join(c for c in "".join(args) if c.isalpha()) if args else "")
    log = open(f"{WORK}/logs/{tag}.log", "a")
    p = subprocess.Popen([sys.executable, "-m", f"flipaware.{mod}", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content/flipaware")
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    p.wait()
    log.close()
    print(f"\n=== stage {n} {mod} finished with code {p.returncode}")
    if p.returncode != 0:
        raise RuntimeError(f"stage {n} failed, see the output above")

import pandas as pd, json
pd.set_option("display.max_colwidth", 200)

def jl(name, folder="data"):
    path = f"{WORK}/{folder}/{name}"
    return pd.read_json(path, lines=True) if os.path.exists(path) and os.path.getsize(path) else pd.DataFrame()

def js(name, folder="results"):
    return json.load(open(f"{WORK}/{folder}/{name}"))
print("helpers ready")

## Stage 0: environment check
Expect `gpu check passed`, your HF username, and the trackers list.

In [ ]:
stage(0, "s00_gpu_check")

## Stage 1: source data
Downloads seed sentences, retrieval pairs, guard pairs, and NevIR. Takes a few minutes.

In [ ]:
stage(1, "s01_data")

In [ ]:
s = jl("seeds.jsonl")
print(len(s), "seeds |", s.has_number.sum(), "with numbers |", s.context.notna().sum(), "with passages")
print(len(jl("human_anchors.jsonl")), "human anchor sentences |", len(jl("nevir_test.jsonl")), "NevIR pairs")
s.sample(5, random_state=0)[["text", "source", "has_number"]]

## Stage 2: generation with Qwen3
Writes a paraphrase and a trap for each seed and trap type. About 10 to 20 minutes on a T4.

In [ ]:
stage(2, "s02_generate", "--set", "main")

In [ ]:
r = jl("raw.jsonl")
print(len(r), "generations |", r.get("skip", pd.Series()).fillna(False).sum(), "skipped |", r.get("error", pd.Series()).fillna(False).sum(), "unparsed")
r.dropna(subset=["paraphrase"]).sample(8, random_state=1)[["type", "anchor", "paraphrase", "trap"]]

## Stage 3: NLI filter
Check the keep rate per trap type. Below 20% for a type means the generator struggles with it.

In [ ]:
stage(3, "s03_filter", "--set", "main")

In [ ]:
st = pd.DataFrame(js("filter_stats.json")).T
st["keep_rate_%"] = (st.kept / st.generated * 100).round(1)
display(st[["generated", "skipped", "unparsed", "paraphrase_not_entailed", "trap_entailed", "trap_low_overlap", "kept", "keep_rate_%"]])
c = jl("candidates.jsonl")
print(c.reason.value_counts())
c[c.kept].sample(6, random_state=2)[["type", "anchor", "paraphrase", "trap", "nli_para", "nli_trap"]]

## Stage 4: splits and passage traps

In [ ]:
stage(4, "s04_splits")

In [ ]:
for f in ("train.jsonl", "dev.jsonl", "test.jsonl", "passage_test.jsonl"):
    print(f, len(jl(f)))
p = jl("passage_test.jsonl")
if len(p):
    x = p.iloc[0]
    print("\nquery:", x.query, "\noriginal:", x.original, "\ntwin:", x.twin)

## Stages 5 and 6: cross generator test set
Phi 4 mini rewrites the held out sentences, then the same filter runs.

In [ ]:
stage(5, "s02_generate", "--set", "xgen")
stage(6, "s03_filter", "--set", "xgen")
print(len(jl("xgen_test.jsonl")), "cross generator test triples")
jl("xgen_test.jsonl").head(5)[["type", "anchor", "paraphrase", "trap"]]

## Stage 7: upload the dataset
Open the printed link and check the dataset card, the Contents table, and the viewer.

In [ ]:
stage(7, "s07_dataset")

## Stage 8: train, upload, test, verdict
Trains MiniLM in 7 variants: headline, 4 ablations, 2 scaling sizes. Look for the `RESULT ... GOOD` or `BAD` block.

In [ ]:
stage(8, "s08_models")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {WORK}/runs

## Stage 9: human test set
Skips with a message until the writer Space has approved triples. That is expected.

In [ ]:
stage(9, "s09_human")

## Stage 10: analysis
Leaderboard models, weight blending, and a refresh of every evaluation.

In [ ]:
stage(10, "s10_analysis")

## Stage 11: final report, cards, leaderboard Space

In [ ]:
stage(11, "s11_finalize")

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{WORK}/results/summary.md").read()))
for img in ("headline.png", "benchmarks.png", "dilution.png", "scaling.png", "blend.png", "leaderboard.png"):
    p = f"{WORK}/assets/{img}"
    if os.path.exists(p):
        print(img); display(Image(p, width=700))

In [ ]:
display(Markdown(open(f"{WORK}/results/REPORT.md").read()))

## Stage 12: publish logs and dashboards

In [ ]:
stage(12, "s12_logs")

In [ ]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{WORK}/tracking/mlflow.db")
runs = mlflow.search_runs(search_all_experiments=True)
cols = [c for c in runs.columns if c in ("tags.mlflow.runName", "metrics.trap/accuracy", "metrics.nanobeir/ndcg10", "metrics.loss", "metrics.train_loss")]
runs[cols]

## Check on HuggingFace

Open each link and confirm it looks right. All `_smoke` repos are private.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
me = api.whoami()["name"]
for kind, items in (("model", api.list_models(author=me, search="_smoke")), ("dataset", api.list_datasets(author=me, search="_smoke")), ("space", api.list_spaces(author=me, search="_smoke"))):
    for it in items:
        prefix = {"model": "", "dataset": "datasets/", "space": "spaces/"}[kind]
        print(f"{kind:8} https://huggingface.co/{prefix}{it.id}")
c = js("collection.json")
print("collection https://huggingface.co/collections/" + c["slug"])

## Optional: deploy the human writer Space
Creates a public Space where people write and check triples. Do this once; it reuses the anchor sentences from stage 1.

In [ ]:
!cd /content/flipaware && python -m flipaware.human deploy

## Optional: pack everything to send for debugging
Zips the logs and results (no models) into your Drive.

In [ ]:
!cd {WORK} && zip -qr /content/drive/MyDrive/flipaware_small_debug.zip logs results tracking/metrics.jsonl -x "*.db" && ls -lh /content/drive/MyDrive/flipaware_small_debug.zip

## Cleanup after the test
Deletes every `_smoke` repo, Space, and dataset, and the smoke collection. Run only when you are done checking.

In [ ]:
CONFIRM = False
if CONFIRM:
    for m in api.list_models(author=me, search="_smoke"):
        api.delete_repo(m.id); print("deleted", m.id)
    for d in api.list_datasets(author=me, search="_smoke"):
        api.delete_repo(d.id, repo_type="dataset"); print("deleted", d.id)
    for s in api.list_spaces(author=me, search="_smoke"):
        api.delete_repo(s.id, repo_type="space"); print("deleted", s.id)
    for c in api.list_collections(owner=me):
        if c.title.endswith("smoke"):
            api.delete_collection(c.slug); print("deleted", c.slug)
else:
    print("set CONFIRM = True to delete")